# S3 Vectors: Metadata Filtering & Benchmark

이 노트북에서는 **Amazon S3 Vectors**에 비디오 임베딩을 저장하고 검색합니다.

S3 Vectors의 핵심 강점은 **설정의 간편함과 비용 효율성**입니다. Vector Bucket과 Index만 생성하면 바로 사용할 수 있고, 별도의 보안 정책이나 VPC 구성 없이 기존 IAM 정책으로 접근 제어가 가능합니다.

이 노트북에서 다루는 내용:
1. 벡터 인제스트 및 속도 측정 (key-based upsert)
2. 메타데이터 필터링 검색
3. 검색 레이턴시 벤치마크 (p50/p95/p99)
4. 결과를 `benchmark_s3vectors.json`에 저장 → 04에서 OpenSearch와 비교

**사전 조건:** `01_setup_and_embeddings.ipynb` 실행 완료

## 0. Setup

In [ ]:
%pip install -r requirements.txt -Uq

In [ ]:
import json, time
import boto3

with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
EMBEDDINGS_S3_PREFIX = config["EMBEDDINGS_S3_PREFIX"]
EMBEDDING_DIMENSION = config["EMBEDDING_DIMENSION"]
VIDEO_IDS = config["VIDEO_IDS"]
S3_VECTOR_BUCKET_FROM_CONFIG = config.get("S3_VECTOR_BUCKET_NAME", "")

session = boto3.Session()
s3 = session.client("s3", region_name=REGION)
s3v = session.client("s3vectors", region_name=REGION)

print(f"✅ 벡터 차원: {EMBEDDING_DIMENSION}, 비디오: {len(VIDEO_IDS)}개")

### ⚙️ Parameters

| 파라미터 | 설명 | 권장 범위 |
|---------|------|----------|
| `VECTOR_BUCKET_NAME` | S3 Vector Bucket 이름 (DB에 해당) | 워크샵 제공 값 사용 |
| `INDEX_NAME` | 벡터 인덱스 이름 (테이블에 해당) | 자유 설정 |
| `INGEST_BATCH_SIZE` | `put_vectors` 호출당 벡터 수. API 최대값은 500 | 50-500 |
| `SEARCH_K` | 검색 시 반환할 최근접 이웃 수 | 5, 10, 20 |

> 📖 **S3 Vectors 구조**: Vector Bucket → Index → Vectors (DB → Table → Rows와 유사)
>
> 각 벡터는 **key** (고유 ID), **data** (임베딩), **metadata** (필터링 가능한 속성)로 구성됩니다. 같은 key로 다시 저장하면 자동 upsert됩니다.

In [ ]:
VECTOR_BUCKET_NAME = S3_VECTOR_BUCKET_FROM_CONFIG or "video-vector-store"
INDEX_NAME = "video-embeddings"
INGEST_BATCH_SIZE = 500
SEARCH_K = 5

## 1. Connect to S3 Vectors

Vector Bucket과 Index를 확인하거나 생성합니다. OpenSearch와 달리 별도의 보안 정책 구성이 필요 없습니다.

> 💡 **OpenSearch와의 차이**: OpenSearch는 암호화/네트워크/데이터 접근 정책 3가지를 별도로 구성해야 하지만, S3 Vectors는 기존 IAM 정책만으로 접근 제어가 가능합니다.

In [ ]:
# Vector Bucket 확인/생성
bucket_exists = any(b["vectorBucketName"] == VECTOR_BUCKET_NAME
                    for b in s3v.list_vector_buckets().get("vectorBuckets", []))
if not bucket_exists:
    s3v.create_vector_bucket(vectorBucketName=VECTOR_BUCKET_NAME)
print(f"✅ Vector Bucket: {VECTOR_BUCKET_NAME} ({'기존' if bucket_exists else '신규 생성'})")

# Index 확인/생성
try:
    s3v.get_index(vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME)
    print(f"✅ Index: {INDEX_NAME} (기존)")
except:
    s3v.create_index(vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME,
                     dimension=EMBEDDING_DIMENSION, distanceMetric="cosine", dataType="float32")
    print(f"✅ Index: {INDEX_NAME} (신규 생성, {EMBEDDING_DIMENSION}차원, cosine)")

## 2. Ingest Embeddings

S3에서 임베딩을 로드하고 S3 Vectors에 인제스트합니다.

> 📖 **배치 크기의 영향**: 블로그 테스트에서 배치 크기를 50→500으로 변경하는 것만으로 인제스트 시간이 약 3배 개선되었습니다 (13.46s → 4.12s). API 최대값인 500을 사용하는 것을 권장합니다.
>
> 📖 **Key-based upsert**: S3 Vectors는 같은 key로 다시 저장하면 자동으로 업데이트됩니다. 임베딩을 재생성해도 별도의 삭제 과정이 필요 없습니다.

In [ ]:
def load_embeddings_from_s3():
    """Load embeddings from S3 (searches multiple paths, handles Workshop 1 and 4 formats)."""
    vectors, found_files = [], []
    for prefix in [EMBEDDINGS_S3_PREFIX, 'embeddings/videos/', 'embeddings/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=200)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('.json') and '/config.json' not in obj['Key']:
                found_files.append(obj['Key'])
        if found_files: break

    # 통합 파일({video_id}.json)이 있으면 output.json/manifest.json 스킵 (중복 방지)
    consolidated = [f for f in found_files if f.split('/')[-1] not in ('output.json', 'manifest.json')]
    if consolidated:
        found_files = consolidated

    for s3_key in found_files:
        try: data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=s3_key)['Body'].read().decode())
        except: continue
        segs = data.get('vectors', data.get('data', []))
        vid = data.get('video_id', '')
        if not vid:
            basename = s3_key.split('/')[-1].replace('.json', '')
            if basename == 'output':
                import os as _os
                s3_uri = data.get('inputVideo', {}).get('s3Uri', '')
                if not s3_uri:
                    s3_uri = data.get('video', {}).get('mediaSource', {}).get('s3Location', {}).get('uri', '')
                if s3_uri:
                    vid = _os.path.splitext(_os.path.basename(s3_uri))[0]
                else:
                    vid = s3_key.split('/')[-3]
            else:
                vid = basename
        for seg in segs:
            if 'embedding' not in seg: continue
            vectors.append({
                'video_id': vid, 'embedding': seg['embedding'],
                'start_sec': seg.get('startSec', seg.get('start_time', 0)),
                'end_sec': seg.get('endSec', seg.get('end_time', 0)),
                'scope': seg.get('embeddingScope', seg.get('embedding_scope', 'clip'))
            })
    return vectors

vectors = load_embeddings_from_s3()
print(f"✅ S3에서 {len(vectors)}개 벡터 로드 완료")

In [ ]:
# Ingest with timing (upsert — 재실행해도 안전)
print(f"   인제스트 중: {len(vectors)}개 벡터, batch_size={INGEST_BATCH_SIZE}...")
ingest_start = time.time()
total = 0
for b in range(0, len(vectors), INGEST_BATCH_SIZE):
    batch = vectors[b:b + INGEST_BATCH_SIZE]
    items = [{
        "key": f"{v['video_id']}_{v['scope']}_{v['start_sec']}_{b+i}",
        "data": {"float32": v["embedding"]},
        "metadata": {"videoId": v["video_id"], "startSec": str(v["start_sec"]),
                     "endSec": str(v["end_sec"]), "scope": v["scope"]}
    } for i, v in enumerate(batch)]
    s3v.put_vectors(vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME, vectors=items)
    total += len(batch)
ingest_time = time.time() - ingest_start

print(f"✅ 인제스트 완료: {total}개 벡터, {ingest_time:.2f}초 소요")

## 3. Metadata Filtering

S3 Vectors의 **메타데이터 필터링**을 실습합니다. 벡터 저장 시 첨부한 메타데이터로 검색 범위를 좁힐 수 있습니다.

| 필터 | 용도 |
|------|------|
| `videoId = "abc"` | 특정 비디오 내에서만 검색 |
| `scope = "clip"` | clip-level 세그먼트만 검색 (asset-level 제외) |

> 📖 **Pre-filter vs Post-filter**: S3 Vectors의 메타데이터 필터링은 **pre-filter**입니다. 유사도 계산 전에 후보를 좁히므로, 항상 k개의 결과를 반환합니다.
>
> 💡 **OpenSearch와의 차이**: OpenSearch는 메타데이터 필터링에 더해 **풀텍스트 검색**(BM25)도 지원합니다. S3 Vectors는 메타데이터의 정확한 값 매칭만 가능합니다.

In [ ]:
# Text-to-embedding
from botocore.config import Config
bedrock = session.client("bedrock-runtime", region_name=REGION, config=Config(signature_version="v4"))

def embed_text(query):
    resp = bedrock.invoke_model(
        modelId="us.twelvelabs.marengo-embed-3-0-v1:0",
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    return json.loads(resp["body"].read())["data"][0]["embedding"]

print("✅ 텍스트 임베딩 함수 준비 완료")

In [ ]:
# ==============================
# 🔎 검색 쿼리를 입력하세요
# ==============================
SEARCH_QUERY = "goal scene in a soccer match"  # ← 변경해보세요!

query_vector = embed_text(SEARCH_QUERY)
print(f"✅ 쿼리: '{SEARCH_QUERY}' → {len(query_vector)}차원 벡터")

In [ ]:
def search_and_print(label, filter_expr=None):
    kwargs = {"vectorBucketName": VECTOR_BUCKET_NAME, "indexName": INDEX_NAME,
              "queryVector": {"float32": query_vector}, "topK": SEARCH_K,
              "returnMetadata": True, "returnDistance": True}
    if filter_expr: kwargs["filter"] = filter_expr
    t0 = time.time()
    result = s3v.query_vectors(**kwargs)
    latency = (time.time() - t0) * 1000
    print(f"\n🔍 {label} (latency={latency:.1f}ms):")
    for i, vec in enumerate(result.get("vectors", []), 1):
        m = vec.get("metadata", {})
        print(f"  {i}. dist={vec.get('distance','N/A'):<8} | {m.get('videoId','?')} | {m.get('startSec','?')}s-{m.get('endSec','?')}s | {m.get('scope','?')}")

# 1) 필터 없음 (기본)
search_and_print("필터 없음")

# 2) clip-level만
search_and_print("clip-level만", {"scope": {"$eq": "clip"}})

# 3) 특정 비디오만
first_video = VIDEO_IDS[0]
search_and_print(f"비디오 '{first_video}'만", {"videoId": {"$eq": first_video}})

In [ ]:
# 비디오 재생
from IPython.display import display, HTML

def build_video_mapping():
    mapping = {}
    for prefix in ['videos/', 'vectordb-workshop/videos/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=200)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('.mp4'):
                import os; mapping[os.path.splitext(os.path.basename(obj['Key']))[0]] = obj['Key']
    for prefix in ['embeddings/videos/', 'vectordb-workshop/embeddings/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=500)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('output.json'):
                try:
                    data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=obj['Key'])['Body'].read().decode())
                    s3_uri = data.get('inputVideo', {}).get('s3Uri', '')
                    if s3_uri:
                        parts = obj['Key'].split('/')
                        mapping[parts[2] if len(parts) > 2 else parts[-2]] = s3_uri.replace(f's3://{S3_BUCKET}/', '')
                except: continue
    return mapping

video_mapping = build_video_mapping()

def play_result(metadata):
    vid = metadata.get('videoId', metadata.get('video_id', ''))
    start = metadata.get('startSec', metadata.get('start_sec', '0'))
    key = video_mapping.get(vid)
    if key:
        url = s3.generate_presigned_url('get_object', Params={'Bucket': S3_BUCKET, 'Key': key}, ExpiresIn=3600)
        display(HTML(f'<p>▶️ {vid} ({start}s~)</p><video width="640" controls><source src="{url}#t={start}" type="video/mp4"></video>'))
    else:
        print(f"⚠️  비디오 미발견: '{vid}'")

# 위 검색 결과의 top 1을 재생 (search_and_print 결과 재사용)
top_result = s3v.query_vectors(
    vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME,
    queryVector={"float32": query_vector}, topK=SEARCH_K,
    returnMetadata=True, returnDistance=True
)
vecs = top_result.get("vectors", [])
if vecs:
    play_result(vecs[0].get("metadata", {}))
else:
    print(f"⚠️  검색 결과 없음")

## 4. Search Benchmark

검색 레이턴시를 측정합니다. 결과는 `benchmark_s3vectors.json`에 저장하여 04에서 OpenSearch와 비교합니다.

> 📖 **블로그 참고 수치**: S3 Vectors는 k=5 기준 p50≈65ms, p95≈181ms로 측정되었습니다. OpenSearch(p50≈25ms)보다 느리지만, 설정 간편함과 비용 효율성이 강점입니다.

In [ ]:
BENCHMARK_ITERATIONS = 20
BENCHMARK_QUERY_COUNT = 5

In [ ]:
import numpy as np

query_vectors_bench = [v["embedding"] for v in vectors if v["scope"] == "asset"][:BENCHMARK_QUERY_COUNT]
if not query_vectors_bench: query_vectors_bench = [vectors[0]["embedding"]]

print(f"벤치마크: {len(query_vectors_bench)}개 쿼리 × {BENCHMARK_ITERATIONS}회 반복\n")

benchmark_results = {"service": "S3 Vectors", "vector_count": len(vectors),
                     "ingest_time": ingest_time, "ingest_batch_size": INGEST_BATCH_SIZE, "search": {}}

for k_val in [5, 10, 20]:
    latencies = []
    for _ in range(BENCHMARK_ITERATIONS):
        for qv in query_vectors_bench:
            t0 = time.time()
            s3v.query_vectors(vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME,
                              queryVector={"float32": qv}, topK=k_val)
            latencies.append((time.time() - t0) * 1000)
    latencies.sort()
    n = len(latencies)
    stats = {"p50": latencies[n//2], "p95": latencies[int(n*0.95)], "p99": latencies[int(n*0.99)], "avg": np.mean(latencies)}
    benchmark_results["search"][f"k={k_val}"] = stats
    print(f"  k={k_val:<3} | p50={stats['p50']:.1f}ms | p95={stats['p95']:.1f}ms | p99={stats['p99']:.1f}ms | avg={stats['avg']:.1f}ms")

print("\n✅ 벤치마크 완료")

In [ ]:
# 결과 저장 (04_comparison.ipynb에서 사용)
with open("benchmark_s3vectors.json", "w") as f:
    json.dump(benchmark_results, f, indent=2)

print(f"✅ benchmark_s3vectors.json 저장 완료")
print(f"   인제스트: {ingest_time:.2f}초 ({len(vectors)}개 벡터, batch={INGEST_BATCH_SIZE})")
print(f"   검색 p50 (k=5): {benchmark_results['search']['k=5']['p50']:.1f}ms")
print(f"\n   다음 단계: 04_comparison.ipynb에서 OpenSearch와 비교")